In [1]:
!pip freeze | grep triton

triton==3.6.0


In [ ]:
@triton.jit 
def get_per_token_logps_kernel(
    logits_ptr,
    input_ids_ptr,
    per_token_logps_ptr,
    vocab_size,
    BLOCK_SIZE: tl.constexpr
): 
    row_id = tl.program_id(0) 
    row_start = row_id * vocab_size

    target_token_id = tl.load(input_ids_ptr + row_id)
    target_token_logit = tl.load(logits_ptr + row_start + target_token_id)

    running_max = float('-inf')
    running_sum = 0
    for i in range(0, vocab_size, BLOCK_SIZE):
        cols = i + tl.arange(0, BLOCK_SIZE)
        mask = cols < vocab_size
        logits_block = tl.load(logits_ptr + row_start + cols, mask=mask, other=float('-inf'))
        m_local = tl.max(logits_block, axis=0)
        d_local = tl.sum(tl.exp(logits_block - m_local), axis=0)
        new_max = tl.maximum(running_max, m_local)
        running_sum = running_sum * tl.exp(running_max - new_max) + d_local * tl.exp(m_local - new_max)
        running_max = new_max
    log_prob = (target_token_logit - running_max) - tl.log(running_sum)
    tl.store(per_token_logps_ptr + row_id, log_prob)


    